# Clase 014 — NumPy: tipos, creación, atributos

**Parte 0** · VanderPlas cap. 2 §§ 2.1-2.2.

> 🎯 Entender por qué `ndarray` es rápido y crear arrays de las 6 formas más usadas.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import sys
import numpy as np
print('numpy:', np.__version__)

## 1️⃣ ¿Por qué `ndarray` es rápido?

Un `list` de Python es **un array de punteros a objetos PyObject**:
```
list = [→PyInt(1), →PyInt(2), →PyInt(3), ...]
```
Cada elemento tiene overhead (~28 bytes en CPython 3.12). Operaciones llaman al intérprete para cada elemento.

Un `ndarray` es **un bloque contiguo de memoria con dtype fijo**:
```
ndarray(int64) = [1, 2, 3, ...]   ← 8 bytes por elemento, sin overhead
```
Operaciones se ejecutan en C, vectorizadas (SIMD).

In [ ]:
import sys
N = 100_000

lst = list(range(N))
arr = np.arange(N)

# Memoria list (sum de getsizeof de cada elemento + overhead lista)
mem_list = sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in lst[:100]) * (N // 100)
mem_arr  = arr.nbytes

print(f'list  : ~{mem_list/1024:.0f} KB')
print(f'array : {mem_arr/1024:.0f} KB')
print(f'ratio : {mem_list/mem_arr:.1f}×')

## 2️⃣ Las 6 formas de crear arrays

VanderPlas las llama "the array creation routines":

In [ ]:
# 1) Desde lista Python
a1 = np.array([1, 2, 3, 4])
print('array:', a1, '  dtype:', a1.dtype)

# 2) Zeros / ones / full (forma y dtype controlados)
print('zeros(5):', np.zeros(5))                        # float64 por default
print('ones((2,3), int):', np.ones((2,3), dtype=int))
print('full((2,2), 7):', np.full((2,2), 7))

# 3) Rangos
print('arange(0,10,2):', np.arange(0, 10, 2))
print('linspace(0,1,5):', np.linspace(0, 1, 5))

# 4) Aleatorios (API moderno con Generator)
rng = np.random.default_rng(seed=42)
print('uniform(3):', rng.random(3))
print('normal(3):', rng.normal(0, 1, 3))

# 5) Identidad y eye
print('eye(3):'); print(np.eye(3))

# 6) Empty (sin inicializar — más rápido, contenido basura)
e = np.empty(3)
print('empty(3):', e, '  ← contenido no inicializado, NO confiar')

## 3️⃣ Atributos: diagnóstico instantáneo

Cuando algo no funciona, **antes de pensar**, mira los atributos:

In [ ]:
M = np.arange(24).reshape(2, 3, 4)
print(f'shape    : {M.shape}')     # (2, 3, 4)
print(f'ndim     : {M.ndim}')      # 3 dimensiones
print(f'size     : {M.size}')      # 24 elementos total
print(f'dtype    : {M.dtype}')
print(f'itemsize : {M.itemsize} bytes/elem')
print(f'nbytes   : {M.nbytes} bytes total')
print(f'strides  : {M.strides}')   # cuánto avanzar en memoria por dim

## 4️⃣ `dtype` — memoria vs precisión

Dtypes principales (VanderPlas tabla 2-1):

| dtype | bytes | rango |
|---|---|---|
| `int8`  | 1 | -128 a 127 |
| `int16` | 2 | -32k a 32k |
| `int32` | 4 | ±2.1e9 |
| `int64` | 8 | ±9.2e18 (default en Linux/macOS, int32 default en Windows < numpy 2.0) |
| `uint8` | 1 | 0 a 255 (imágenes RGB) |
| `float32` | 4 | ±3.4e38, ~7 dígitos precisión |
| `float64` | 8 | ±1.7e308, ~15 dígitos (default) |
| `bool` | 1 | True/False |

**Elige `float32`** cuando trabajes con redes neuronales en GPU (la mitad de memoria, suficiente precisión para gradientes).

## 5️⃣ ⚠️ Bug clásico: overflow silencioso

NumPy no levanta excepción cuando un dtype no alcanza — *wrap-around* silencioso:

In [ ]:
# Overflow demo
a = np.array([100, 200, 50], dtype=np.int8)
print('original:', a)
b = a + 200
print('+200    :', b, '  ← ¡debería ser [300, 400, 250]!')
print('  por qué: int8 va de -128 a 127, los valores hicieron wrap-around')

# Fix: dtype suficiente o promoción explícita
c = a.astype(np.int32) + 200
print('fix     :', c)

## 6️⃣ Random reproducible — API moderno

```python
# ❌ legacy (deprecated en favor del nuevo Generator)
np.random.seed(42)
np.random.rand(5)

# ✅ moderno
rng = np.random.default_rng(seed=42)
rng.random(5)
rng.normal(0, 1, 5)
rng.integers(0, 10, 5)
```

Ventajas: independiente entre instancias (puedes tener varios rngs), algoritmo más rápido (PCG64), API más limpio.

In [ ]:
# Mismo seed → mismo output (reproducibilidad)
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)
print(rng_a.normal(0, 1, 5))
print(rng_b.normal(0, 1, 5))   # idéntico
print('iguales?', np.array_equal(rng_a.normal(0,1,5), rng_b.normal(0,1,5)))

## ✅ Checklist

- [ ] Sé por qué ndarray es más rápido que list
- [ ] Conozco las 6 formas de crear arrays
- [ ] Sé inspeccionar shape, dtype, ndim, nbytes
- [ ] Anticipé overflow al elegir dtype
- [ ] Uso `np.random.default_rng(seed)` no `np.random.seed`

## 📝 Homework

Ver `README.md`. Memoria list vs ndarray, 6 formas, overflow int8, función `info(arr)`.

## 🔗 Referencias

- VanderPlas cap. 2 §§ 2.1-2.2
- [Array creation](https://numpy.org/doc/stable/user/basics.creation.html)
- [Generator API](https://numpy.org/doc/stable/reference/random/generator.html)

➡️ **Siguiente:** [015 — NumPy: ufuncs y vectorización](../015-numpy-ufuncs-y-vectorizacion/README.md)